First, load the package and instantiate the dependencies.

In [1]:
using Pkg
pkg"activate .; instantiate; precompile" # may take a long time the first time this is run

  Updating registry at `~/.julia/registries/General`
  Updating git-repo `https://github.com/JuliaRegistries/General.git`
Precompiling project...


And load the deps

In [2]:
using CSV, Plots, JSON, BenchmarkTools, Parameters, PerlaTonettiWaugh
gr(fmt = :png) # for plot displays

Plots.GRBackend()

## Stationary Solutions

First, define common objects for the full model

In [3]:
parameters = parameter_defaults(); # model parameters
settings = settings_defaults(run_global = false); # numerical solution settings

To solve for time 0 and ``T`` stationary solutions, simply:

In [4]:
params_0 = parameter_defaults(d = parameters.d_0)
params_T = parameter_defaults(d = parameters.d_T)
stationary_sol_0 = stationary_algebraic(params_0, settings);
stationary_sol_T = stationary_algebraic(params_T, settings);

We can inspect each:

In [5]:
foreach(println, pairs(stationary_sol_0))

Pair{Symbol,Any}(:F, #F#32{Float64}(5.0018))
Pair{Symbol,Any}(:r, 0.0482417)
Pair{Symbol,Any}(:ν, 1.51378)
Pair{Symbol,Any}(:a, 10.9077)
Pair{Symbol,Any}(:b, 0.602576)
Pair{Symbol,Any}(:S, 0.0989518)
Pair{Symbol,Any}(:L_tilde, 0.230576)
Pair{Symbol,Any}(:L_tilde_x, 0.0209138)
Pair{Symbol,Any}(:L_tilde_E, 0.114541)
Pair{Symbol,Any}(:L_tilde_a, 0.0951211)
Pair{Symbol,Any}(:z_bar, 1.34291)
Pair{Symbol,Any}(:w, 0.919614)
Pair{Symbol,Any}(:x, 1.0)
Pair{Symbol,Any}(:π_min, 0.186648)
Pair{Symbol,Any}(:π_rat, 1.81101)
Pair{Symbol,Any}(:g, 0.00674174)
Pair{Symbol,Any}(:z_hat, 1.97755)
Pair{Symbol,Any}(:Ω, 0.961287)
Pair{Symbol,Any}(:y, 1.03327)
Pair{Symbol,Any}(:c, 1.03327)
Pair{Symbol,Any}(:λ_ii, 0.895606)
Pair{Symbol,Any}(:U_bar, 16.1068)


And

In [6]:
foreach(println, pairs(stationary_sol_T))

Pair{Symbol,Any}(:F, #F#32{Float64}(5.0018))
Pair{Symbol,Any}(:r, 0.0500704)
Pair{Symbol,Any}(:ν, 1.42423)
Pair{Symbol,Any}(:a, 10.4546)
Pair{Symbol,Any}(:b, 0.525821)
Pair{Symbol,Any}(:S, 0.108098)
Pair{Symbol,Any}(:L_tilde, 0.235066)
Pair{Symbol,Any}(:L_tilde_x, 0.0284007)
Pair{Symbol,Any}(:L_tilde_E, 0.10836)
Pair{Symbol,Any}(:L_tilde_a, 0.0983055)
Pair{Symbol,Any}(:z_bar, 1.33558)
Pair{Symbol,Any}(:w, 0.914597)
Pair{Symbol,Any}(:x, 1.0)
Pair{Symbol,Any}(:π_min, 0.187777)
Pair{Symbol,Any}(:π_rat, 1.82894)
Pair{Symbol,Any}(:g, 0.00857042)
Pair{Symbol,Any}(:z_hat, 1.83968)
Pair{Symbol,Any}(:Ω, 0.909407)
Pair{Symbol,Any}(:y, 1.02163)
Pair{Symbol,Any}(:c, 1.02163)
Pair{Symbol,Any}(:λ_ii, 0.857402)
Pair{Symbol,Any}(:U_bar, 19.5362)


## Transition Dynamics

The code is

In [7]:
cachename = join(hash((parameters = parameters, settings = settings))) # uniquely identify the run by parameters + settings
cachepath = joinpath(pwd(), "data", cachename * ".csv")
use_cache = true # change this to false if you don't want to use the cache for a run
write_cache = true # change this to false if you don't want to generate *any* cache files for a run

# reading
if isfile(cachepath) && use_cache
    println("using cache.")
    solved = CSV.read(cachepath);
else
    println("solving without cache.")
    @time solution = solve_transition(parameters, settings);
    solved = solution.results;
end

# writing
if write_cache
    mkpath(joinpath(pwd(), "data")) # if this doesn't eist for whatever reason
    CSV.write(cachepath, solved) # write DataFrame
    write(joinpath(pwd(), "data", cachename * ".json"), JSON.json((parameters = parameters, settings = merge(settings, (interp = typeof(settings.interp),))))) # write parameters
else
    # nothing
end

solving without cache.
 37.158428 seconds (91.34 M allocations: 18.041 GiB, 15.56% gc time)


12578